<a href="https://colab.research.google.com/github/alanwarr/BCS_Vibe_Consulting_Event/blob/main/Vibe_Consulting_Demo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Client Use of LLM As a Consultant (DIY Vibe Consulting)

This is to simulate a niave client with a consulting type question.  

Practically such a client would use a Chat UI and may employ some prompting techniques.

We will use three AI models from different vendors.

The **question** our naive client will ask is:

*We’re a UK mid-market retailer with falling online conversion over the last 3 months. What should we do in the next 90 days to diagnose and fix this?*

Let us see how the frontier LLMs perform? How will they compare with a human consultant?

####Install packages

In [1]:
!pip -q install --upgrade anthropic google-generativeai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.5/357.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.8 MB/s eta 0:00:00


####Set API Keys for:

*   ChatGPT 4 from OpenAI
*   Claude from Anthropic
*   Gemini from Google

In [2]:
import os, getpass
os.environ["OPENAI_API_KEY"]   = getpass.getpass("OpenAI API key: ")
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
os.environ["GOOGLE_API_KEY"]    = getpass.getpass("Google (AI Studio) API key: ")

OpenAI API key: ··········
Anthropic API key: ··········
Google (AI Studio) API key: ··········


####Choosing the Models

In [3]:
TEMPERATURE = 0.3
MAX_TOKENS  = 2000

# OpenAI
OPENAI_PREFERRED = ["gpt-4.1", "gpt-4o", "gpt-4.1-mini"]

# Anthropic — try rolling aliases first, then older versioned IDs
ANTHROPIC_PREFERRED = [
    "claude-3-5-sonnet-latest",
    "claude-3-5-haiku-latest",
    "claude-3-5-sonnet-20240620",
    "claude-3-opus-20240229",
]

In [4]:
from anthropic import Anthropic
anth = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def call_anthropic(question: str, models=ANTHROPIC_PREFERRED):
    last_err = None
    for m in models:
        try:
            t0 = time.time()
            resp = anth.messages.create(
                model=m,
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                messages=[{"role":"user","content":question}],
            )
            t1 = time.time()
            text = "\n".join(b.text for b in resp.content if getattr(b, "type", None) == "text").strip()
            usage = getattr(resp, "usage", None)
            in_tok  = getattr(usage, "input_tokens", None) if usage else None
            out_tok = getattr(usage, "output_tokens", None) if usage else None
            return {
                "provider":"Anthropic","model":m,"latency_s":round(t1-t0,2),
                "input_tokens":in_tok,"output_tokens":out_tok,"answer":text
            }
        except Exception as e:
            last_err = e
            continue
    raise last_err

In [5]:
def provider_status():
    print("OpenAI models (first preference available):", OPENAI_PREFERRED[0])
    print("Anthropic preference order:", ANTHROPIC_PREFERRED)

    # Gemini 2.x via google.genai
    from google import genai
    try:
        client = genai.Client()
        models = client.models.list()
        model_names = [m.name for m in models]
        print("Gemini models available:", model_names[:8])
    except Exception as e:
        print("Gemini list_models error:", e)

provider_status()

OpenAI models (first preference available): gpt-4.1
Anthropic preference order: ['claude-3-5-sonnet-latest', 'claude-3-5-haiku-latest', 'claude-3-5-sonnet-20240620', 'claude-3-opus-20240229']
Gemini models available: ['models/embedding-gecko-001', 'models/gemini-2.5-pro-preview-03-25', 'models/gemini-2.5-flash-preview-05-20', 'models/gemini-2.5-flash', 'models/gemini-2.5-flash-lite-preview-06-17', 'models/gemini-2.5-pro-preview-05-06', 'models/gemini-2.5-pro-preview-06-05', 'models/gemini-2.5-pro']


####Set Up Three Models for Single Query without any Prompt Engineering

In [15]:
import time
import pandas as pd
from tabulate import tabulate

# ----------------- OpenAI -----------------
from openai import OpenAI
oai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def call_openai(question: str, models=OPENAI_PREFERRED):
    last_err = None
    t0 = time.time()
    for m in models:
        try:
            resp = oai.chat.completions.create(
                model=m,
                messages=[{"role": "user", "content": question}],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
            )
            t1 = time.time()
            text  = resp.choices[0].message.content.strip()
            usage = getattr(resp, "usage", None)
            in_tok  = getattr(usage, "prompt_tokens", None) if usage else None
            out_tok = getattr(usage, "completion_tokens", None) if usage else None
            return {"provider":"OpenAI","model":m,"latency_s":round(t1-t0,2),
                    "input_tokens":in_tok,"output_tokens":out_tok,"answer":text}
        except Exception as e:
            last_err = e
    raise last_err

# ----------------- Anthropic -----------------
from anthropic import Anthropic
anth = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def call_anthropic(question: str, models=ANTHROPIC_PREFERRED):
    last_err = None
    for m in models:
        try:
            t0 = time.time()
            resp = anth.messages.create(
                model=m,
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                messages=[{"role":"user","content":question}],
            )
            t1 = time.time()
            text = "\n".join(b.text for b in resp.content if getattr(b,"type",None)=="text").strip()
            usage = getattr(resp, "usage", None)
            in_tok  = getattr(usage, "input_tokens", None) if usage else None
            out_tok = getattr(usage, "output_tokens", None) if usage else None
            return {"provider":"Anthropic","model":m,"latency_s":round(t1-t0,2),
                    "input_tokens":in_tok,"output_tokens":out_tok,"answer":text}
        except Exception as e:
            last_err = e
            continue
    raise last_err

# ----------------- Google Gemini (v2.5, new SDK) -----------------

import time
from google import genai

# Client auto-reads GOOGLE_API_KEY from your environment / Colab secrets
gemini_client = genai.Client()

# Pick one model for the demo (fast + reliable)
GEMINI_MODEL = "gemini-2.5-flash"      # or "gemini-2.5-pro" for deeper reasoning

def call_gemini(question: str, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    t0 = time.time()
    resp = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=question,
        config={
            "temperature": float(temperature),
            # Removed max_output_tokens here
            }
    )

    t1 = time.time()

    text = "" # Initialize text as empty
    in_tok = None
    out_tok = None

    # Robustly check for content and extract text
    if resp and resp.candidates:
        candidate = resp.candidates[0] # Assuming we only care about the first candidate
        if getattr(candidate, 'content', None) and getattr(candidate.content, 'parts', None):
             # Join text from all parts if available
            text = "".join(getattr(part, 'text', '') for part in candidate.content.parts)

        # Attempt to get token counts if usage_metadata is available
        if getattr(resp, 'usage_metadata', None):
            in_tok = getattr(resp.usage_metadata, "prompt_token_count", None)
            out_tok = getattr(resp.usage_metadata, "candidates_token_count", None)

        # If text is still empty despite candidate/content/parts existing,
        # or if finishReason is not STOP and text is empty, raise an error
        # Added more specific check: text is empty AND content/parts were expected
        if not text.strip() and getattr(candidate.content, 'parts', None):
             # If parts exist but yielded no text, something is wrong
            raise ValueError("Gemini API returned content structure but no text.")

        # Optional: Raise error for non-STOP finish reasons if text is empty
        # finish_reason = getattr(candidate, 'finish_reason', None)
        # if finish_reason and finish_reason.name != 'STOP' and not text.strip():
        #     raise ValueError(f"Gemini API returned non-STOP finish reason ({finish_reason.name}) with no text.")

    else:
         # If no candidates were returned at all
         raise ValueError("Gemini API returned no candidates.")


    return {
        "provider": "Google",
        "model": GEMINI_MODEL,
        "latency_s": round(t1 - t0, 2),
        "input_tokens": in_tok,
        "output_tokens": out_tok,
        "answer": text.strip()
    }

print("✅ Gemini ready with:", GEMINI_MODEL)

✅ Gemini ready with: gemini-2.5-flash


####A Function to Run All Three Models, Side-by-Side

In [10]:
# ----------------- Run All Three Models Side-by-Side -----------------
import pandas as pd
from tabulate import tabulate

def run_benchmark(question: str, providers=("openai", "anthropic", "google")):
    """
    Runs the same consultancy-style question across the selected LLM providers
    (OpenAI, Anthropic, Google Gemini) and prints a side-by-side comparison
    of latency, model name, and token usage (where available).
    """
    print("\n🔍 Running benchmark for question:\n\n", question, "\n\n")

    rows = []
    answers = []

    # --- OpenAI ---
    if "openai" in providers:
        try:
            r = call_openai(question)
            rows.append([
                r["provider"], r["model"], r["latency_s"],
                r.get("input_tokens"), r.get("output_tokens")
            ])
            answers.append(("OpenAI", r["answer"]))
        except Exception as e:
            rows.append(["OpenAI", "(error)", "—", "—", "—"])
            answers.append(("OpenAI", f"⚠️ Error: {e}"))

    # --- Anthropic ---
    if "anthropic" in providers:
        try:
            r = call_anthropic(question)
            rows.append([
                r["provider"], r["model"], r["latency_s"],
                r.get("input_tokens"), r.get("output_tokens")
            ])
            answers.append(("Anthropic", r["answer"]))
        except Exception as e:
            rows.append(["Anthropic", "(error)", "—", "—", "—"])
            answers.append(("Anthropic", f"⚠️ Error: {e}"))

    # --- Google Gemini (v2.5, via new google.genai client) ---
    if "google" in providers:
        try:

            # Call the call_gemini function, which returns a dictionary
            r = call_gemini(question)
            rows.append([
                r["provider"], r["model"], r["latency_s"],
                r.get("input_tokens"), r.get("output_tokens")
            ])
            answers.append(("Google Gemini", r["answer"]))
        except Exception as e:
            rows.append(["Google Gemini", "(error)", "—", "—", "—"])
            answers.append(("Google Gemini", f"⚠️ Error: {e}"))

    # --- Format the results ---
    df = pd.DataFrame(rows, columns=[
        "Provider", "Model", "Latency (s)", "Input tokens", "Output tokens"
    ])
    print(tabulate(df, headers="keys", tablefmt="github", showindex=False))

    print("\n" + "=" * 90 + "\n")
    for name, ans in answers:
        print(f"## {name}\n\n\n{ans.strip()}\n" + "-" * 90 + "\n")

####Client Query - Simple ask, demanding problem, no prompt design

In [16]:
question_example = (
    "We’re a UK mid-market retailer with falling online conversion over the last 3 months. "
    "What should we do in the next 90 days to diagnose and fix this?"
)

run_benchmark(question_example)


🔍 Running benchmark for question:

 We’re a UK mid-market retailer with falling online conversion over the last 3 months. What should we do in the next 90 days to diagnose and fix this? 


| Provider   | Model                   |   Latency (s) |   Input tokens |   Output tokens |
|------------|-------------------------|---------------|----------------|-----------------|
| OpenAI     | gpt-4.1                 |         14.65 |             41 |             766 |
| Anthropic  | claude-3-5-haiku-latest |          7.61 |             46 |             376 |
| Google     | gemini-2.5-flash        |         35.77 |             38 |            2326 |


## OpenAI


Absolutely, here’s a practical 90-day action plan to **diagnose and address falling online conversion** for a UK mid-market retailer:

---

## **Weeks 1–2: Diagnose the Problem**

### **1. Data Analysis**
- **Review Analytics:** Deep-dive into Google Analytics (or equivalent). Look for:
  - Where in the funnel drop-offs are happening 